# 03A – Model Training (Enterprise)

This notebook trains the production bankruptcy prediction model using the **preprocessed dataset** `american_bankruptcy_cleaned.csv`.

## Business Objective
Train a production-ready machine learning model for bankruptcy prediction using the cleaned dataset and save all deployment artifacts.

In [1]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
DATA_PATH=r"C:\Users\suvha\Desktop\BANkruptcy RISK PREDICTION\bankruptcy-early-warning-screener\data\datasets\american_bankruptcy_cleaned.csv"

df=pd.read_csv(DATA_PATH)

target='status_label' if 'status_label' in df.columns else 'target'

if target=='status_label':
    y=df[target].map({'alive':0,'failed':1})
else:
    y=df[target]

X=df.drop(columns=[target])

X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Training Shape:',X_train.shape)
print('Testing Shape:',X_test.shape)

Training Shape: (62945, 20)
Testing Shape: (15737, 20)


In [3]:
model=RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

model.fit(X_train,y_train)

joblib.dump(model,'production_bankruptcy_model.joblib')
print('Model saved successfully.')

ValueError: could not convert string to float: 'C_4569'

In [ ]:
pred=model.predict(X_test)
prob=model.predict_proba(X_test)[:,1]

metrics=pd.DataFrame({
    'Metric':['Accuracy','Precision','Recall','F1 Score','ROC-AUC'],
    'Value':[
        accuracy_score(y_test,pred),
        precision_score(y_test,pred),
        recall_score(y_test,pred),
        f1_score(y_test,pred),
        roc_auc_score(y_test,prob)
    ]
})

metrics.to_csv('training_metrics.csv',index=False)
metrics

In [ ]:
feature_importance=pd.DataFrame({
    'Feature':X.columns,
    'Importance':model.feature_importances_
}).sort_values('Importance',ascending=False)

feature_importance.to_csv('feature_importance.csv',index=False)

plt.figure(figsize=(10,8))
plt.barh(
    feature_importance.head(20)['Feature'][::-1],
    feature_importance.head(20)['Importance'][::-1]
)
plt.title('Top 20 Feature Importances')
plt.tight_layout()
plt.savefig('feature_importance.png',dpi=300)
plt.show()

## Deliverables

- `production_bankruptcy_model.joblib`
- `training_metrics.csv`
- `feature_importance.csv`
- `feature_importance.png`

These artifacts are used by the Explainability (Phase 4), Validation (Phase 5), and Deployment (Phase 6) notebooks.

## Executive Summary

A production Random Forest model has been trained using the cleaned dataset. The trained model and supporting artifacts are exported for downstream explainability, validation, and deployment workflows.